In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.utils import murmurhash3_32

from tiger.config import Config

In [2]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 2070


In [ ]:
cfg = Config.from_yaml()

In [3]:
seq = pd.read_parquet("./data/interim/2014/sequences_Beauty.parquet")
sem = pd.read_parquet("./data/processed/2014/Beauty_semantic_ids.parquet")
items = pd.read_parquet("./data/interim/2014/meta_Beauty.parquet")

In [ ]:
asin_to_idx = {asin: i for i, asin in enumerate(items["asin"].tolist())}

idx_to_sem_id = {}
for idx, raw_sem_id in enumerate(sem.to_dict(orient="tight")["data"]):
    proc_sem_id = [level * cfg.rqvae.codebook_size + sem_token for level, sem_token in enumerate(raw_sem_id)]
    idx_to_sem_id[idx] = proc_sem_id

asin_to_sem_id = {asin: idx_to_sem_id[asin_to_idx[asin]] for asin in list(asin_to_idx.keys())}

In [ ]:
def to_semantic_sequence(user_id: str, asin_seq: list[str], num_bins: int):
    offset = (cfg.rqvae.num_codebooks + 1) * cfg.rqvae.codebook_size
    user_token_id = murmurhash3_32(user_id, positive=True) % num_bins + offset

    sem_seq: np.ndarray = np.array([user_token_id], dtype=np.int64)
    for asin in asin_seq:
        sem_id = asin_to_sem_id[asin]
        sem_seq = np.append(sem_seq, sem_id)
    return sem_seq

In [ ]:
sem_seqs = seq.apply(lambda x: to_semantic_sequence(x["reviewerID"], x["asin"], cfg.t5.user_num_bins), axis=1)

In [7]:
with open("./data/processed/2014/Beauty_semantic_sequences.npy", "wb") as f:
    np.save(f, sem_seqs)

In [ ]:
min_len = float("inf")
max_len = -float("inf")
for sem_seq in sem_seqs:
    min_len = min(sem_seq.shape[0], min_len)
    max_len = max(sem_seq.shape[0], max_len)

print(f"Sequences lengths are in the range [{min_len}, {max_len}]")

assert min_len == (cfg.rqvae.num_codebooks + 1) * (cfg.dataset.min_reviews_per_user) + 1
assert max_len == (cfg.rqvae.num_codebooks + 1) * cfg.dataset.sequence_length + 1

Sequences lengths are in the range [21, 81]
